# Book Price Tracker — Web Scraping 

This notebook documents the process of building a web scraper that collects
book data (title, price, availability, rating) from
[books.toscrape.com](https://books.toscrape.com/), a public sandbox site
built specifically for scraping practice — no restrictions apply here.

The final, production-ready version of this project is available as a single
script: `book_price_tracker.py`. This notebook walks through **how** that
script was built, step by step, including the libraries used, why each piece
of code is needed, and a data-quality issue that came up along the way and
how it was fixed.

**Libraries used:** `requests`, `beautifulsoup4`, `pandas`


## Step 1 — Fetch a page and inspect the raw HTML

Before extracting anything, we send a request to the target page and confirm
we're getting a valid response. `response.status_code == 200` confirms the
request succeeded, and setting `response.encoding` explicitly to `"utf-8"`
avoids garbled special characters (like the £ sign) later on.


In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"
response = requests.get(url)
response.encoding = "utf-8"

print(response.status_code)
print(response.text[:500])


200
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


## Step 2 — Parse the HTML and locate each product block

`BeautifulSoup` turns the raw HTML string into a navigable tree structure.
`find_all("article", class_="product_pod")` then pulls out every HTML block
that represents a single book on the page (found by inspecting the page's
HTML structure in the browser dev tools).


In [2]:
soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")
print(len(books))
print(books[0])


20
<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>


## Step 3 — Extract structured data from a single book

From one product block, we pull out four fields:

- **title** — read from the link's `title` attribute (the visible text is
  truncated, e.g. "A Light in the...")
- **price** — the visible text of the `price_color` element
- **availability** — the visible text of the `instock availability` element,
  cleaned up with `.strip()` to remove extra whitespace/newlines
- **rating** — stored as a CSS class (e.g. `"star-rating Three"`), so we pull
  the second class name from the tag's `class` list


In [3]:
book = books[0]

title = book.h3.a["title"]
price = book.find("p", class_="price_color").text
availability = book.find("p", class_="instock availability").text.strip()
rating = book.find("p", class_="star-rating")["class"][1]

print(title)
print(price)
print(availability)
print(rating)


A Light in the Attic
£51.77
In stock
Three


## Step 4 — Scale up to every book on the page

The same extraction logic is applied to all 20 books on the page in a loop,
collecting each book as a dictionary and appending it to a list. This
list-of-dictionaries format converts cleanly into a pandas DataFrame.


In [4]:
data = []

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    availability = book.find("p", class_="instock availability").text.strip()
    rating = book.find("p", class_="star-rating")["class"][1]

    data.append({
        "title": title,
        "price": price,
        "availability": availability,
        "rating": rating,
    })

print(len(data))
data[0]


20


{'title': 'A Light in the Attic',
 'price': '£51.77',
 'availability': 'In stock',
 'rating': 'Three'}

## Step 5 — Handle pagination to scrape the entire catalog

A single page only covers 20 of the ~1000 books on the site. The catalog
pages follow a predictable URL pattern (`page-1.html`, `page-2.html`, ...),
so the scraping logic is wrapped into two reusable functions:

- `scrape_page(url)` — scrapes one page and returns its book data
- `scrape_all_pages(base_url, total_pages)` — loops through every page,
  collects the results, and:
  - wraps each request in `try/except` so a single failed page doesn't
    crash the whole run
  - adds a short `time.sleep()` delay between requests to avoid hammering
    the server

Note: at this stage `product_url` (each book's own link) is also captured —
this turns out to be important in Step 8.


In [5]:
import time

def scrape_page(url):
    """Scrapes a single page and returns its book data as a list of dicts."""
    response = requests.get(url)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")
    page_data = []

    for book in books:
        title = book.h3.a["title"]
        product_url = book.h3.a["href"]
        price = book.find("p", class_="price_color").text.replace("£", "")
        availability = book.find("p", class_="instock availability").text.strip()
        rating = book.find("p", class_="star-rating")["class"][1]

        page_data.append({
            "title": title,
            "product_url": product_url,
            "price": float(price),
            "availability": availability,
            "rating": rating,
        })

    return page_data


def scrape_all_pages(base_url, total_pages):
    """Loops through every page and combines the results into one list."""
    all_data = []

    for page_num in range(1, total_pages + 1):
        url = base_url.format(page_num)
        try:
            page_data = scrape_page(url)
            all_data.extend(page_data)
            print(f"Page {page_num} done — {len(page_data)} books")
        except Exception as e:
            print(f"Page {page_num} failed: {e}")

        time.sleep(1)

    return all_data


In [6]:
import pandas as pd

base_url = "https://books.toscrape.com/catalogue/page-{}.html"
all_data = scrape_all_pages(base_url, total_pages=50)

df = pd.DataFrame(all_data)
print(len(df))
df.head()


Page 1 done — 20 books
Page 2 done — 20 books
Page 3 done — 20 books
Page 4 done — 20 books
Page 5 done — 20 books
Page 6 done — 20 books
Page 7 done — 20 books
Page 8 done — 20 books
Page 9 done — 20 books
Page 10 done — 20 books
Page 11 done — 20 books
Page 12 done — 20 books
Page 13 done — 20 books
Page 14 done — 20 books
Page 15 done — 20 books
Page 16 done — 20 books
Page 17 done — 20 books
Page 18 done — 20 books
Page 19 done — 20 books
Page 20 done — 20 books
Page 21 done — 20 books
Page 22 done — 20 books
Page 23 done — 20 books
Page 24 done — 20 books
Page 25 done — 20 books
Page 26 done — 20 books
Page 27 done — 20 books
Page 28 done — 20 books
Page 29 done — 20 books
Page 30 done — 20 books
Page 31 done — 20 books
Page 32 done — 20 books
Page 33 done — 20 books
Page 34 done — 20 books
Page 35 done — 20 books
Page 36 done — 20 books
Page 37 done — 20 books
Page 38 done — 20 books
Page 39 done — 20 books
Page 40 done — 20 books
Page 41 done — 20 books
Page 42 done — 20 books
P

,title,product_url,price,availability,rating
0,A Light in the Attic,a-light-in-the-attic_1000/index.html,51.77,In stock,Three
1,Tipping the Velvet,tipping-the-velvet_999/index.html,53.74,In stock,One
2,Soumission,soumission_998/index.html,50.10,In stock,One
3,Sharp Objects,sharp-objects_997/index.html,47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,sapiens-a-brief-history-of-humankind_996/index...,54.23,In stock,Five


## Step 6 — Clean the data and save a snapshot

`df.info()` confirms there are no missing values and that `price` is stored
as `float64` (not text) — needed for any numeric comparison later. The
cleaned snapshot is then saved to a dated CSV file, so future runs can be
compared against it.


In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         1000 non-null   object 
 1   product_url   1000 non-null   object 
 2   price         1000 non-null   float64
 3   availability  1000 non-null   object 
 4   rating        1000 non-null   object 
dtypes: float64(1), object(4)
memory usage: 39.2+ KB


In [8]:
import os
from datetime import date

def save_snapshot(df, folder="price_history"):
    """Saves a CSV file tagged with today's date and returns its path."""
    os.makedirs(folder, exist_ok=True)
    today = date.today().isoformat()
    filepath = f"{folder}/books_{today}.csv"
    df.to_csv(filepath, index=False)
    print(f"Saved: {filepath}")
    return filepath

today_path = save_snapshot(df)


Saved: price_history/books_2026-09-20.csv


## Step 7 — Build the price-comparison logic

To detect price changes over time, a new snapshot is compared against the
most recent previous one using `pandas.merge()` — matching rows from both
tables on a shared key, similar to a SQL join.

The first version below matches on `"title"`.


In [9]:
def compare_prices(old_path, new_df, key="title"):
    """Compares a previous snapshot against new data and returns changed prices."""
    old_df = pd.read_csv(old_path)

    merged = old_df.merge(new_df, on=key, suffixes=("_old", "_new"))
    merged["price_change"] = merged["price_new"] - merged["price_old"]
    changed = merged[merged["price_change"] != 0]

    return changed


## Step 8 — Data-quality check: is `title` actually a safe key to match on?

Before trusting `"title"` as the join key, it's worth checking whether titles
are actually unique across the catalog. This is a good general habit before
merging two tables on any column.


In [10]:
duplicate_titles = df[df.duplicated("title", keep=False)]
print(f"{duplicate_titles['title'].nunique()} title(s) appear more than once in the catalog")
duplicate_titles.sort_values("title").head()


1 title(s) appear more than once in the catalog


,title,product_url,price,availability,rating
236,The Star-Touched Queen,the-star-touched-queen_764/index.html,46.02,In stock,Five
358,The Star-Touched Queen,the-star-touched-queen_642/index.html,32.30,In stock,Five


**Result:** some titles are *not* unique — a handful of books share the same
name but are different products (different prices, different pages). Merging
on `"title"` in that case creates incorrect matches: pandas pairs up **every**
old row with **every** new row that shares that title, producing false price
"changes" that don't correspond to real books.

**Fix:** match on `"product_url"` instead, which uniquely identifies each
book regardless of shared titles.


In [11]:
def compare_prices(old_path, new_df):
    """Compares a previous snapshot against new data and returns changed prices.

    Matching is done on 'product_url' rather than 'title', because the site
    can contain multiple books with the same title — matching on title alone
    would produce incorrect (cartesian) matches.
    """
    old_df = pd.read_csv(old_path)

    merged = old_df.merge(new_df, on="product_url", suffixes=("_old", "_new"))
    merged["price_change"] = merged["price_new"] - merged["price_old"]
    changed = merged[merged["price_change"] != 0]

    return changed[["title_new", "price_old", "price_new", "price_change"]]


## Step 9 — Test the fix with an artificial price change

Since prices on this sandbox site never actually change, a few prices are
modified artificially on a copy of the data to confirm the comparison logic
correctly detects changes — and only the real ones.


In [12]:
df_new = df.copy()
df_new.loc[0, "price"] = df_new.loc[0, "price"] + 5.00
df_new.loc[1, "price"] = df_new.loc[1, "price"] - 2.50
df_new.loc[2, "price"] = df_new.loc[2, "price"] + 10.00

changes = compare_prices(today_path, df_new)
changes


,title_new,price_old,price_new,price_change
0,A Light in the Attic,51.77,56.77,5.0
1,Tipping the Velvet,53.74,51.24,-2.5
2,Soumission,50.10,60.10,10.0


With the fix in place, only the three books that were actually changed show
up — no false matches from duplicate titles.

## Next steps

This notebook covers the exploration and reasoning behind the approach. The
cleaned-up, reusable version of this logic — organized into functions with
error handling, a `main()` entry point, and no exploratory/debugging code —
lives in **`book_price_tracker.py`**, ready to run as a standalone script or
scheduled task.
